# 统一视角：八个自由度与三条可检验的判断

对应文章：《大模型量化算法（23）：LLM PTQ 统一视角》
https://lrypcy.github.io/2026/08/29/llm-quant-23-unified-view/

八个自由度（F1~F8），本文动手实现其中五个：

| | 自由度 | 本文的实现 |
|---|---|---|
| F1 | 网格类型 | 均匀整数（本文默认；NF4 见 `fp8_mxfp4_formats`） |
| F2 | 网格位置与间距（scale） | 逐输出通道 scale 的网格搜索 |
| F3 | 粒度 | per-channel（本文默认；粒度实验见 `quantizer_granularity`） |
| F4 | 落点规则（rounding） | **F4a = GPTQ 误差补偿**，**F4b = AdaRound 学舍入** |
| F5 | 等效变换 | **F5a = SmoothQuant**，**F5b = AWQ 式激活感知缩放**（同一个自由度的两种配方） |
| F7 | 混合精度 | 见 `mixed_precision` |
| F8 | 模型适应 | QAT 微调（STE 训练权重去适配网格） |

要检验的三条判断：

- **判断一**：同类自由度叠加收益递减，异类叠加收益可加
- **判断二**：bit 下降时发力点必须沿 F1 → F7 → F8 迁移
- **判断四**：动 F5（恒等变换）的方法部署最顺——因为它能**融合进权重**

统一评测口径：一个线性层 $y=XW^\top$，激活 per-tensor 8-bit、权重 per-channel $b$-bit，
指标是**测试集上的输出相对误差**（不是权重误差，也不是校准集误差）。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"
CFG = {
    "smoke": dict(d_in=48, d_out=48, n_cal=1024, n_test=512, bits=(8, 4, 3, 2),
                  a_bits=8, n_alpha=21, n_gamma=15, n_f2=41, ar_passes=4, qat_steps=300),
    "full":  dict(d_in=96, d_out=96, n_cal=4096, n_test=2048, bits=(8, 6, 4, 3, 2),
                  a_bits=8, n_alpha=41, n_gamma=25, n_f2=61, ar_passes=6, qat_steps=900),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
def qmax_of(b):
    return 2 ** (b - 1) - 1
def rel_err(a, b):
    return float(np.linalg.norm(a - b) / np.linalg.norm(a))
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'d_in': 48, 'd_out': 48, 'n_cal': 1024, 'n_test': 512, 'bits': (8, 4, 3, 2), 'a_bits': 8, 'n_alpha': 21, 'n_gamma': 15, 'n_f2': 41, 'ar_passes': 4, 'qat_steps': 300}


## 0 · 合成层与量化算子

激活带**离群通道**（真实 LLM 的形态），权重带**重尾**。这让 F5（等效变换）有真实用武之地，
也让「per-channel 的 max 标定」明显不是最优（给 F2 留出空间）。

In [2]:
rng = np.random.default_rng(SEED)
d_in, d_out = CFG["d_in"], CFG["d_out"]
n_cal, n_test = CFG["n_cal"], CFG["n_test"]
A_BITS = CFG["a_bits"]

X_cal = rng.normal(0, 1, (n_cal, d_in))
out_ch = rng.choice(d_in, max(2, d_in // 12), replace=False)
X_cal[:, out_ch] *= 10.0                                   # 离群激活通道
X_test = rng.normal(0, 1, (n_test, d_in)); X_test[:, out_ch] *= 10.0

W = rng.normal(0, 1, (d_out, d_in)) / np.sqrt(d_in)        # (d_out, d_in)
W[rng.random(W.shape) < 0.03] *= 6.0                       # 权重重尾
W[:, out_ch] *= 0.2                                        # 离群通道配对的权重反而小
Y_test = X_test @ W.T
H = X_cal.T @ X_cal / n_cal

log(f"层形状 W={W.shape}   校准 {n_cal}   测试 {n_test}")
log(f"激活 absmax/rms = {np.abs(X_cal).max()/np.sqrt((X_cal**2).mean()):.1f}  (离群通道)")
log(f"权重 absmax/rms = {np.abs(W).max()/np.sqrt((W**2).mean()):.1f}  (重尾)")

def quant_act(A, b=A_BITS):
    s = np.abs(A).max() / qmax_of(b)
    return np.round(A / s) * s

def rtn(W2, b, scale=None):
    if scale is None:
        scale = np.abs(W2).max(axis=1, keepdims=True) / qmax_of(b)
    return np.round(W2 / scale) * scale

def eval_Wq(Wq, X_te=None):
    X_te = X_test if X_te is None else X_te
    return rel_err(Y_test, quant_act(X_te) @ Wq.T)

RTN = {}
log("\n基线 RTN（WbA%d）：" % A_BITS)
for b in CFG["bits"]:
    RTN[b] = eval_Wq(rtn(W, b))
    log(f"  W{b}A{A_BITS}: 输出相对误差 = {RTN[b]:.4f}")

层形状 W=(48, 48)   校准 1024   测试 512
激活 absmax/rms = 15.6  (离群通道)
权重 absmax/rms = 9.9  (重尾)

基线 RTN（WbA8）：
  W8A8: 输出相对误差 = 0.0732
  W4A8: 输出相对误差 = 0.3064
  W3A8: 输出相对误差 = 0.4904
  W2A8: 输出相对误差 = 0.6914


## 1 · 五个自由度各自的边际收益

- **F2（scale）**：固定 round-to-nearest，逐输出通道网格搜索 scale 的缩放因子
  $\gamma\in[0.5,1.5]$，最小化校准集输出误差。LSQ / MSE-observer 那一族在做的事。
- **F4a（GPTQ）**：按输入维顺序量化，$H=X^\top X$，用 $H^{-1}$ 把当前列的量化误差分摊到**尚未量化**的列。
- **F4b（AdaRound）**：从 RTN 出发，逐个权重在 floor/ceil 之间做坐标下降，
  解析判据 $\Delta\mathcal{L}=-2\delta\,(DH)_{o,i}+\delta^2 H_{i,i}$（$D=W-W_q$）。
- **F5a（SmoothQuant）**：$s_j=\max|X_j|^\alpha/\max|W_{:,j}|^{1-\alpha}$。
- **F5b（AWQ 式）**：$s_j=(\overline{|X_j|}/\overline{\overline{|X|}})^\gamma$，只用激活幅度。
  两者都是**同一个自由度（per-input-channel 等效变换）的不同配方**。
- **F8（QAT）**：STE 训练权重本身去适配网格，并把权重夹在网格范围内防漂移。

In [3]:
B_MAIN = 4

# ---------- F2：scale 网格搜索 ----------
def f2_optimize(W2, b, n_grid=None):
    n_grid = n_grid or CFG["n_f2"]
    base = np.abs(W2).max(axis=1, keepdims=True) / qmax_of(b)
    best = np.full(W2.shape[0], np.inf)
    bg = np.ones((W2.shape[0], 1))
    for g in np.linspace(0.5, 1.5, n_grid):
        s = base * g
        wq = np.round(W2 / s) * s
        e = np.sum((X_cal @ W2.T - X_cal @ wq.T) ** 2, axis=0)   # 逐输出通道独立
        m = e < best
        best = np.where(m, e, best)
        bg = np.where(m[:, None], g, bg)
    return np.round(W2 / (base * bg)) * (base * bg)

# ---------- F4a：GPTQ 误差补偿 ----------
def gptq(W2, H_, b, damp=0.01):
    d_i = W2.shape[1]; qmb = qmax_of(b)
    Hd = H_ + damp * np.mean(np.diag(H_)) * np.eye(d_i)
    L = np.linalg.cholesky(Hd); Hinv = np.linalg.inv(L.T) @ np.linalg.inv(L)
    s = np.abs(W2).max(axis=1, keepdims=True) / qmb
    Wc = W2.copy(); Wq = np.zeros_like(W2)
    for i in range(d_i):
        w = Wc[:, i]
        q = np.clip(np.round(w / s[:, 0]), -qmb, qmb) * s[:, 0]
        Wq[:, i] = q
        err = (w - q) / Hinv[i, i]
        if i + 1 < d_i:
            Wc[:, i + 1:] -= np.outer(err, Hinv[i, i + 1:])
    return Wq

# ---------- F4b：AdaRound 坐标下降 ----------
def adaround(W2, H_, b, passes=None):
    passes = passes or CFG["ar_passes"]
    qmb = qmax_of(b)
    s = np.abs(W2).max(axis=1, keepdims=True) / qmb
    Wq = np.round(W2 / s) * s
    for _ in range(passes):
        G = (W2 - Wq) @ H_
        dg = np.diag(H_); improved = 0
        for o in range(W2.shape[0]):
            ss = s[o, 0]; xv = W2[o] / ss
            lo = np.floor(xv) * ss; hi = np.ceil(xv) * ss
            for i in range(W2.shape[1]):
                cur = Wq[o, i]
                alt = hi[i] if abs(cur - lo[i]) <= abs(cur - hi[i]) else lo[i]
                delta = alt - cur
                if abs(delta) < 1e-15:
                    continue
                dL = -2.0 * delta * G[o, i] + delta * delta * dg[i]
                if dL < -1e-18:
                    Wq[o, i] = alt
                    G[o, :] -= delta * H_[:, i]
                    improved += 1
        if improved == 0:
            break
    return Wq

# ---------- F5：等效变换（同一个自由度的两种配方）----------
def s_smooth(alpha):
    mx = np.abs(X_cal).max(axis=0); mw = np.abs(W).max(axis=0)
    return (mx ** alpha) / np.maximum(mw ** (1 - alpha), 1e-12)

def s_awq(gamma):
    ax = np.abs(X_cal).mean(axis=0)
    return (ax / max(ax.mean(), 1e-12)) ** gamma

def eval_transform(s_vec, b, Wq_fn=None):
    """X -> X/s, W -> W*s，乘积严格不变；只改量化难度"""
    Wq_fn = Wq_fn or (lambda M: rtn(M, b))
    return rel_err(Y_test, quant_act(X_test / s_vec) @ Wq_fn(W * s_vec).T)

def best_smooth(b):
    al = np.linspace(0.0, 1.0, CFG["n_alpha"])
    vals = [eval_transform(s_smooth(a), b) for a in al]
    i = int(np.argmin(vals)); return float(al[i]), float(vals[i]), np.array(vals)

def best_awq(b):
    gs = np.linspace(0.0, 2.0, CFG["n_gamma"])
    vals = [eval_transform(s_awq(g), b) for g in gs]
    i = int(np.argmin(vals)); return float(gs[i]), float(vals[i]), np.array(vals)

# ---------- F8：QAT 微调（STE + 夹在网格内）----------
LAM = float(np.linalg.norm(X_cal, 2) ** 2 / n_cal)
def qat_finetune(W2, b, steps=None, lr_scale=0.02):
    """STE 训练 W 去适配网格；权重夹在网格内防漂移；按校准损失做早停（best-of-trajectory）"""
    steps = steps or CFG["qat_steps"]
    lr = lr_scale / LAM
    qmb = qmax_of(b); s = np.abs(W2).max(axis=1, keepdims=True) / qmb
    Wt = W2.copy(); Xq = quant_act(X_cal); Yref = Xq @ W2.T
    best = (np.inf, None)
    for _ in range(steps):
        Wq = np.clip(np.round(Wt / s) * s, -qmb * s, qmb * s)
        cal = float(np.sum((Xq @ Wq.T - Yref) ** 2))
        if cal < best[0]:
            best = (cal, Wq.copy())
        Wt = np.clip(Wt - lr * ((Xq.T @ (Xq @ Wq.T - Yref)).T / n_cal), -qmb * s, qmb * s)
    return best[1]

al_star, e_f5a, curve_a = best_smooth(B_MAIN)
g_star, e_f5b, curve_g = best_awq(B_MAIN)
single = {
    "RTN": RTN[B_MAIN],
    "F2": eval_Wq(f2_optimize(W, B_MAIN)),
    "F4a": eval_Wq(gptq(W, H, B_MAIN)),
    "F4b": eval_Wq(adaround(W, H, B_MAIN)),
    "F5a": e_f5a,
    "F5b": e_f5b,
    "F8": eval_Wq(qat_finetune(W, B_MAIN)),
}
log(f"\n=== 1 单自由度边际收益 @W{B_MAIN}A{A_BITS}（基线 RTN = {RTN[B_MAIN]:.4f}）===")
log(f"{'自由度':<8}{'输出相对误差':>16}{'相对 RTN 增益':>16}")
for k in ["RTN", "F2", "F4a", "F4b", "F5a", "F5b", "F8"]:
    log(f"{k:<8}{single[k]:>16.4f}{(1-single[k]/RTN[B_MAIN])*100:>15.1f}%")
log(f"\n  SmoothQuant 最优 alpha = {al_star:.2f}   AWQ 最优 gamma = {g_star:.2f}")
log("  读数：F5（等效变换）是这一层上收益最大的一维——因为它改的是『问题』本身；")
log("        F2（scale）也有实打实的收益；而 F4（落点规则）在这一层上几乎没有收益，")
log("        原因见下节注释：per-channel + RTN 已经很接近该网格上的最优落点。")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ks = ["RTN", "F2", "F4a", "F4b", "F5a", "F5b", "F8"]
ax[0].bar(ks, [single[k] for k in ks], color=["#8c8c8c", "#4c72b0", "#dd8452", "#c44e52",
                                              "#55a868", "#8172b2", "#937860"])
ax[0].axhline(RTN[B_MAIN], color="black", ls="--", lw=1, label="RTN baseline")
ax[0].set_ylabel("output rel error"); ax[0].legend(fontsize=8)
ax[0].set_title(f"[1] single-DOF gains @W{B_MAIN}A{A_BITS}"); ax[0].grid(alpha=0.3, axis="y")
ax[1].plot(np.linspace(0, 1, CFG["n_alpha"]), curve_a, marker="o", ms=3, label="F5a SmoothQuant (alpha)")
ax[1].plot(np.linspace(0, 2, CFG["n_gamma"]), curve_g, marker="s", ms=3, label="F5b AWQ (gamma)")
ax[1].axhline(RTN[B_MAIN], color="black", ls="--", lw=1, label="RTN")
ax[1].set_xlabel("transform strength"); ax[1].set_ylabel("output rel error")
ax[1].legend(fontsize=8); ax[1].set_title("[1] two recipes of the same DOF"); ax[1].grid(alpha=0.3)
savefig(fig, "uv_a_single_dof.png")


=== 1 单自由度边际收益 @W4A8（基线 RTN = 0.3064）===
自由度               输出相对误差       相对 RTN 增益
RTN               0.3064            0.0%
F2                0.2161           29.5%
F4a               0.3087           -0.8%
F4b               0.3119           -1.8%
F5a               0.1482           51.6%
F5b               0.1776           42.0%
F8                0.3065           -0.1%

  SmoothQuant 最优 alpha = 0.45   AWQ 最优 gamma = 0.86
  读数：F5（等效变换）是这一层上收益最大的一维——因为它改的是『问题』本身；
        F2（scale）也有实打实的收益；而 F4（落点规则）在这一层上几乎没有收益，
        原因见下节注释：per-channel + RTN 已经很接近该网格上的最优落点。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_a_single_dof.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_a_single_dof.png'

## 2 · 判断一：同类叠加递减，异类叠加可加

判据：设 $\Delta_m = 1 - e_m/e_{\mathrm{RTN}}$ 是方法 $m$ 单独用的相对增益，
叠加后的实测增益为 $\Delta_{12}$。定义

$$\text{叠加效率} = \frac{\Delta_{12}}{\Delta_{m_1}+\Delta_{m_2}}$$

- **同类（F5a ⊕ F5b）**：两者都是「per-input-channel 等效变换」的配方。联合搜索
  $s = s_{\text{smooth}}(\alpha)\cdot s_{\text{awq}}(\gamma)$ 的二维网格——
  它们抢的是同一个自由度，所以联合搜索拿不到「两者收益之和」。
- **异类（F5a ⊕ F2）**：F5 改**问题**（等效变换后的新分布），F2 在新问题上解**网格间距**——互不冲突。

这就是「工业方案总是挑每类里最好的一个，而不是挑精度最高的三个」的定量依据。

In [4]:
def gain(e, b=B_MAIN):
    return 1.0 - e / RTN[b]

# ---- 同类：F5a ⊕ F5b（联合二维搜索同一个等效变换）----
joint = (np.inf, None, None)
for a in np.linspace(0.0, 1.0, CFG["n_alpha"]):
    for g in np.linspace(0.0, 2.0, CFG["n_gamma"]):
        e = eval_transform(s_smooth(a) * s_awq(g), B_MAIN)
        if e < joint[0]:
            joint = (e, float(a), float(g))
e_joint = float(joint[0])

# ---- 异类：F5a ⊕ F2 ----
s_a = s_smooth(al_star)
e_f5a_f2 = rel_err(Y_test, quant_act(X_test / s_a) @ f2_optimize(W * s_a, B_MAIN).T)
s_b = s_awq(g_star)
e_f5b_f2 = rel_err(Y_test, quant_act(X_test / s_b) @ f2_optimize(W * s_b, B_MAIN).T)

# ---- 参考：F4a ⊕ F4b（都在 F4）：从 GPTQ 的解出发再做 AdaRound ----
Wq_g = gptq(W, H, B_MAIN)
def adaround_from(W2, H_, b, W_start, passes=None):
    passes = passes or CFG["ar_passes"]
    qmb = qmax_of(b)
    s = np.abs(W2).max(axis=1, keepdims=True) / qmb
    Wq = W_start.copy()
    for _ in range(passes):
        G = (W2 - Wq) @ H_
        dg = np.diag(H_); improved = 0
        for o in range(W2.shape[0]):
            ss = s[o, 0]; xv = W2[o] / ss
            lo = np.floor(xv) * ss; hi = np.ceil(xv) * ss
            for i in range(W2.shape[1]):
                cur = Wq[o, i]
                alt = hi[i] if abs(cur - lo[i]) <= abs(cur - hi[i]) else lo[i]
                delta = alt - cur
                if abs(delta) < 1e-15:
                    continue
                dL = -2.0 * delta * G[o, i] + delta * delta * dg[i]
                if dL < -1e-18:
                    Wq[o, i] = alt
                    G[o, :] -= delta * H_[:, i]
                    improved += 1
        if improved == 0:
            break
    return Wq

e_f4_both = eval_Wq(adaround_from(W, H, B_MAIN, Wq_g))

combos = [
    ("F5a + F5b  (同类)", single["F5a"], single["F5b"], e_joint),
    ("F5a + F2   (异类)", single["F5a"], single["F2"], e_f5a_f2),
    ("F5b + F2   (异类)", single["F5b"], single["F2"], e_f5b_f2),
    ("F4a + F4b  (同类)", single["F4a"], single["F4b"], e_f4_both),
]
log(f"=== 2 判断一：叠加效率 @W{B_MAIN}A{A_BITS}（基线 RTN={RTN[B_MAIN]:.4f}）===")
log(f"{'组合':<20}{'Δ单独1':>10}{'Δ单独2':>10}{'Δ之和':>10}{'Δ叠加':>10}{'叠加效率':>10}")
rows_2 = []
for name, e1, e2, e12 in combos:
    d1, d2, d12 = gain(e1), gain(e2), gain(e12)
    denom = d1 + d2
    eff = d12 / denom if abs(denom) > 1e-9 else float("nan")
    rows_2.append(dict(combo=name, d1=d1, d2=d2, dsum=denom, d12=d12, eff=eff))
    log(f"{name:<20}{d1:>10.3f}{d2:>10.3f}{denom:>10.3f}{d12:>10.3f}{eff:>10.3f}")
log("-" * 78)
same = [r for r in rows_2 if "同类" in r["combo"]]
cross = [r for r in rows_2 if "异类" in r["combo"]]
log(f"  读数：同类叠加效率 = {[round(r['eff'],3) for r in same]} —— 明显 < 1，它们争抢同一个自由度；")
log(f"        异类叠加效率 = {[round(r['eff'],3) for r in cross]} —— 接近或超过 1，互不冲突。")
log(f"        （F4a+F4b 这一对在本探针上 Δ 本身接近 0：per-channel 网格上 RTN 已近最优落点，")
log( "          GPTQ/AdaRound 只能在『H 估计噪声』里找便宜，泛化到测试集后归零甚至略负。）")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
short = ["F5a+F5b\n(same)", "F5a+F2\n(cross)", "F5b+F2\n(cross)", "F4a+F4b\n(same)"]
xs = np.arange(len(rows_2))
ax[0].bar(xs - 0.2, [r["dsum"] for r in rows_2], 0.4, label="sum of individual gains")
ax[0].bar(xs + 0.2, [r["d12"] for r in rows_2], 0.4, label="actual combined gain")
ax[0].set_xticks(xs); ax[0].set_xticklabels(short, fontsize=7)
ax[0].set_ylabel("relative gain over RTN"); ax[0].legend(fontsize=8)
ax[0].set_title("[2] same-family stacking is sub-additive"); ax[0].grid(alpha=0.3, axis="y")
ax[1].bar(short, [r["eff"] for r in rows_2],
          color=["#c44e52", "#55a868", "#55a868", "#c44e52"])
ax[1].axhline(1.0, color="grey", ls="--", lw=1.2, label="perfectly additive")
ax[1].set_ylabel("stacking efficiency"); ax[1].legend(fontsize=8)
ax[1].set_title("[2] efficiency = combined / (sum of parts)"); ax[1].grid(alpha=0.3, axis="y")
ax[1].tick_params(axis="x", labelsize=7)
savefig(fig, "uv_b_stacking_efficiency.png")

=== 2 判断一：叠加效率 @W4A8（基线 RTN=0.3064）===
组合                        Δ单独1      Δ单独2       Δ之和       Δ叠加      叠加效率
F5a + F5b  (同类)          0.516     0.420     0.937     0.516     0.551
F5a + F2   (异类)          0.516     0.295     0.811     0.729     0.899
F5b + F2   (异类)          0.420     0.295     0.715     0.684     0.957
F4a + F4b  (同类)         -0.008    -0.018    -0.026    -0.014     0.539
------------------------------------------------------------------------------
  读数：同类叠加效率 = [0.551, 0.539] —— 明显 < 1，它们争抢同一个自由度；
        异类叠加效率 = [0.899, 0.957] —— 接近或超过 1，互不冲突。
        （F4a+F4b 这一对在本探针上 Δ 本身接近 0：per-channel 网格上 RTN 已近最优落点，
          GPTQ/AdaRound 只能在『H 估计噪声』里找便宜，泛化到测试集后归零甚至略负。）


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_b_stacking_efficiency.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_b_stacking_efficiency.png'

## 3 · 判断二：bit 下降时，发力点必须迁移

对每个权重 bit，分别测 F2 / F4a / F4b / F5a / F5b / F8 单独能捞回多少（相对该 bit 的 RTN 基线）。
文章预测：4-bit 时优化 F2/F3/F4 收益巨大；3-bit 时 F7 开始成为必需；
2-bit 时 F1 或 F8 几乎是唯一出路——**还在优化 F2 的 scale 注定失败**。

In [5]:
log("=== 3 判断二：各 bit 下每个自由度的输出相对误差 ===")
keys = ["RTN", "F2", "F4a", "F4b", "F5a", "F5b", "F8"]
log(f"{'bits':>6}" + "".join(f"{k:>10}" for k in keys))
rows_3, gain3 = [], {}
for b in CFG["bits"]:
    r = {"bits": b, "RTN": RTN[b], "F2": eval_Wq(f2_optimize(W, b)),
         "F4a": eval_Wq(gptq(W, H, b)), "F4b": eval_Wq(adaround(W, H, b)),
         "F5a": best_smooth(b)[1], "F5b": best_awq(b)[1],
         "F8": eval_Wq(qat_finetune(W, b))}
    rows_3.append(r)
    gain3[b] = {k: (1 - r[k] / r["RTN"]) * 100 for k in keys[1:]}
    log(f"{b:>6}" + "".join(f"{r[k]:>10.4f}" for k in keys))

log("\n相对该 bit 的 RTN 基线的增益（%）:")
log(f"{'bits':>6}" + "".join(f"{k:>10}" for k in keys[1:]))
for r in rows_3:
    log(f"{r['bits']:>6}" + "".join(f"{gain3[r['bits']][k]:>10.1f}" for k in keys[1:]))
log("-" * 78)
g8, g4, g2 = gain3[8], gain3[4], gain3[2]
log(f"  读数：F2（scale）的收益从 8-bit 的 {g8['F2']:.1f}% / 4-bit 的 {g4['F2']:.1f}% "
    f"掉到 2-bit 的 {g2['F2']:.1f}% —— 网格间距这一维被榨干了；")
log(f"        F5（等效变换）的收益也从 {g8['F5a']:.1f}% 掉到 {g2['F5a']:.1f}%。")
log("        所有『在固定网格里做最优』的自由度收益都在随 bit 下降而塌缩 ->")
log("        低 bit 必须换赛道：F7（保住 outlier）/ F1（非均匀网格）/ F8（改被量化的函数）。")
log(f"        F8 在本探针上基本等价于 RTN（增益 {gain3[4]['F8']:+.1f}%）：单层线性层 + 校准与测试")
log("        同分布时，RTN 已经落在网格上的近最优位置，没有可『适应』的空间。")
log("        F8 的真实价值在多层误差累积 / 校准-推理分布漂移的场景——这正呼应判断二：")
log("        『2-bit 必须换到 F8』：那时可适配的空间才真正出现。")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k in keys[1:]:
    ax[0].plot([r["bits"] for r in rows_3], [gain3[r["bits"]][k] for r in rows_3], marker="o", label=k)
ax[0].axhline(0, color="grey", lw=0.8)
ax[0].set_xlabel("weight bits"); ax[0].set_ylabel("gain over RTN (%)")
ax[0].set_title("[3] where the leverage moves as bits drop"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[0].invert_xaxis()
bot = np.zeros(len(rows_3))
for k in keys[1:]:
    v = np.array([max(gain3[r["bits"]][k], 0) for r in rows_3])
    ax[1].bar([str(r["bits"]) for r in rows_3], v, bottom=bot, label=k)
    bot += v
ax[1].set_xlabel("weight bits"); ax[1].set_ylabel("stacked available gain (%)")
ax[1].set_title("[3] composition of available gain"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, axis="y")
savefig(fig, "uv_c_bit_migration.png")

=== 3 判断二：各 bit 下每个自由度的输出相对误差 ===
  bits       RTN        F2       F4a       F4b       F5a       F5b        F8


     8    0.0732    0.0684    0.0731    0.0731    0.0134    0.0140    0.0731


     4    0.3064    0.2161    0.3087    0.3119    0.1482    0.1776    0.3065


     3    0.4904    0.3362    0.4933    0.4952    0.3253    0.3649    0.4918


     2    0.6914    0.5560    0.6914    0.6918    0.6199    0.6200    0.6923

相对该 bit 的 RTN 基线的增益（%）:
  bits        F2       F4a       F4b       F5a       F5b        F8
     8       6.6       0.2       0.2      81.7      80.9       0.1
     4      29.5      -0.8      -1.8      51.6      42.0      -0.1
     3      31.4      -0.6      -1.0      33.7      25.6      -0.3
     2      19.6       0.0      -0.1      10.3      10.3      -0.1
------------------------------------------------------------------------------
  读数：F2（scale）的收益从 8-bit 的 6.6% / 4-bit 的 29.5% 掉到 2-bit 的 19.6% —— 网格间距这一维被榨干了；
        F5（等效变换）的收益也从 81.7% 掉到 10.3%。
        所有『在固定网格里做最优』的自由度收益都在随 bit 下降而塌缩 ->
        低 bit 必须换赛道：F7（保住 outlier）/ F1（非均匀网格）/ F8（改被量化的函数）。
        F8 在本探针上基本等价于 RTN（增益 -0.1%）：单层线性层 + 校准与测试
        同分布时，RTN 已经落在网格上的近最优位置，没有可『适应』的空间。
        F8 的真实价值在多层误差累积 / 校准-推理分布漂移的场景——这正呼应判断二：
        『2-bit 必须换到 F8』：那时可适配的空间才真正出现。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_c_bit_migration.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_c_bit_migration.png'

## 4 · 判断四：动 F5 的方法部署最顺

F5 是**恒等变换**：$X'=X/s,\ W'=W\cdot s$，于是 $X'W'^\top = XW^\top$ **严格成立**，
$s$ 可以直接**融合进上一层的 LayerNorm / Linear 权重**，推理时零额外算子、零额外延迟。

反例：F1（非均匀网格如 NF4）输出不是均匀整数，只能**查表**还原；
F7（混合精度）要稀疏 kernel 或多路径调度——「精度漂亮但落地困难」的方法最集中在那一维。

In [6]:
s_vec = s_smooth(al_star)
fp_out = X_cal @ W.T
tf_out = (X_cal / s_vec) @ (W * s_vec).T
max_abs_diff = float(np.abs(fp_out - tf_out).max())
scale_ref = float(np.abs(fp_out).max())
log("=== 4 判断四：F5 是严格恒等变换 ===")
log(f"  ||X W^T - (X/s)(W s)^T||_max = {max_abs_diff:.3e}   "
    f"（相对输出量级 {max_abs_diff/scale_ref:.3e}）")
log(f"  -> 融合前：推理图上多一次 per-channel elementwise 乘")
log(f"  -> 融合后：s 折进上一层权重，这一步从推理图上完全消失，等价性误差 = 浮点舍入级")
log(f"  迁移向量 s 的动态范围 = {s_vec.max()/s_vec.min():.1f}x（越接近 1 说明两边难度本来就均衡）")

deploy = [
    ("F5 等效变换 (SmoothQuant/AWQ)", "折叠进相邻层权重", "0", "无"),
    ("F2/F3 scale & 粒度", "per-channel scale", "极低", "int kernel 原生支持"),
    ("F4 落点 (GPTQ/AdaRound)", "离线改权重值", "0", "无（离线完成）"),
    ("F1 非均匀网格 (NF4)", "查表 dequant", "中", "需 LUT / gather"),
    ("F7 混合精度", "每层不同 kernel", "中-高", "kernel 分派 + 多 dequant 路径"),
    ("F8 模型适应 (QAT)", "需训练", "离线高", "推理无额外代价"),
]
log("\n部署代价清单:")
for a, b_, c, d_ in deploy:
    log(f"  {a:<32}{b_:<22}{c:<8}{d_}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].semilogy(np.arange(s_vec.size), np.sort(s_vec)[::-1], marker="o", ms=3)
ax[0].set_xlabel("input channel (sorted)"); ax[0].set_ylabel("migration scale s_j")
ax[0].set_title(f"[4] SmoothQuant migration vector (alpha={al_star:.2f})"); ax[0].grid(alpha=0.3)
ax[1].bar(["FP32 y", "after F5 transform"], [scale_ref, float(np.abs(tf_out).max())],
          color=["#4c72b0", "#55a868"])
ax[1].set_ylabel("max |y|"); ax[1].set_title(f"[4] identity holds to {max_abs_diff/scale_ref:.1e}")
ax[1].grid(alpha=0.3, axis="y")
savefig(fig, "uv_d_deployability.png")

=== 4 判断四：F5 是严格恒等变换 ===
  ||X W^T - (X/s)(W s)^T||_max = 2.665e-15   （相对输出量级 2.138e-16）
  -> 融合前：推理图上多一次 per-channel elementwise 乘
  -> 融合后：s 折进上一层权重，这一步从推理图上完全消失，等价性误差 = 浮点舍入级
  迁移向量 s 的动态范围 = 18.7x（越接近 1 说明两边难度本来就均衡）

部署代价清单:
  F5 等效变换 (SmoothQuant/AWQ)       折叠进相邻层权重              0       无
  F2/F3 scale & 粒度                per-channel scale     极低      int kernel 原生支持
  F4 落点 (GPTQ/AdaRound)           离线改权重值                0       无（离线完成）
  F1 非均匀网格 (NF4)                  查表 dequant            中       需 LUT / gather
  F7 混合精度                         每层不同 kernel           中-高     kernel 分派 + 多 dequant 路径
  F8 模型适应 (QAT)                   需训练                   离线高     推理无额外代价


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_d_deployability.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_d_deployability.png'

## 结论汇总

In [7]:
summary = {
    "meta": {"mode": MODE, "seed": SEED, "layer": [d_out, d_in],
             "n_cal": n_cal, "n_test": n_test, "act_bits": A_BITS,
             "main_weight_bits": B_MAIN},
    "1_single_dof": {k: float(v) for k, v in single.items()},
    "1_params": {"smooth_alpha": al_star, "awq_gamma": g_star},
    "2_stacking": rows_2,
    "3_bit_migration": {"abs_err": [{k: float(v) for k, v in r.items()} for r in rows_3],
                        "gain_pct": {str(b): {k: float(v) for k, v in g.items()}
                                     for b, g in gain3.items()}},
    "4_identity": {"max_abs_diff": max_abs_diff, "rel": max_abs_diff / scale_ref,
                   "s_dynamic_range": float(s_vec.max() / s_vec.min())},
}
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES))
print("\n".join(_LINES[-8:]))
print("\n[done] results.json + stdout.txt written")


部署代价清单:
  F5 等效变换 (SmoothQuant/AWQ)       折叠进相邻层权重              0       无
  F2/F3 scale & 粒度                per-channel scale     极低      int kernel 原生支持
  F4 落点 (GPTQ/AdaRound)           离线改权重值                0       无（离线完成）
  F1 非均匀网格 (NF4)                  查表 dequant            中       需 LUT / gather
  F7 混合精度                         每层不同 kernel           中-高     kernel 分派 + 多 dequant 路径
  F8 模型适应 (QAT)                   需训练                   离线高     推理无额外代价
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/unified_view/results/uv_d_deployability.png

[done] results.json + stdout.txt written
